In [1]:
import pandas as pd
import numpy as np

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag


from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# sample text data 

text = [
    "Payment failed while making transaction",
    "Unable to login in my account",
    "Payment deducted but order not placed",
    "Account hacked and password changed",
    "Login issue after password reset",
    "Machine learning is very interesting",
    "Deep learning is powerful and useful",
    "NLP helps in text processing"
]

# Converting into DataFrame
df = pd.DataFrame({"Text": text})

print("Original Text Data:")
print(df)

Original Text Data:
                                      Text
0  Payment failed while making transaction
1            Unable to login in my account
2    Payment deducted but order not placed
3      Account hacked and password changed
4         Login issue after password reset
5     Machine learning is very interesting
6     Deep learning is powerful and useful
7             NLP helps in text processing


In [3]:
# Initializing stopwords and lemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [4]:
# Function to get correct POS tag for lemmatization

def get_pos(word):
    tag = pos_tag([word])[0][1][0].upper()

    tag_dict = {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "V": wordnet.VERB,
        "R": wordnet.ADV
    }

    return tag_dict.get(tag, wordnet.NOUN)

In [5]:
# Preprocessing function

def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    
    filtered_words = [word for word in tokens if word.isalpha() and word not in stop_words]
    lemmatized_words = [lemmatizer.lemmatize(word, get_pos(word)) for word in filtered_words]
    
    clean_text = " ".join(lemmatized_words)
    return clean_text

In [6]:
# Applying preprocessing on the text column

df["Processed_Text"] = df["Text"].apply(preprocess_text)

print("After Pre-processing:")
print(df)

After Pre-processing:
                                      Text                 Processed_Text
0  Payment failed while making transaction  payment fail make transaction
1            Unable to login in my account           unable login account
2    Payment deducted but order not placed     payment deduct order place
3      Account hacked and password changed   account hack password change
4         Login issue after password reset     login issue password reset
5     Machine learning is very interesting         machine learn interest
6     Deep learning is powerful and useful     deep learn powerful useful
7             NLP helps in text processing       nlp help text processing


In [7]:
# Feature Engineering using TF-IDF

vectorizer = TfidfVectorizer()

X_tfidf = vectorizer.fit_transform(df["Processed_Text"])

print("TF-IDF Matrix Shape:")
print(X_tfidf.shape)

TF-IDF Matrix Shape:
(8, 25)


In [9]:
# Convert TF-IDF matrix into DataFrame

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print("TF-IDF Feature Table:")
print(tfidf_df)

TF-IDF Feature Table:
    account    change    deduct      deep      fail      hack  help  interest  \
0  0.000000  0.000000  0.000000  0.000000  0.519708  0.000000   0.0  0.000000   
1  0.540443  0.000000  0.000000  0.000000  0.000000  0.000000   0.0  0.000000   
2  0.000000  0.000000  0.519708  0.000000  0.000000  0.000000   0.0  0.000000   
3  0.454195  0.541948  0.000000  0.000000  0.000000  0.541948   0.0  0.000000   
4  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   0.0  0.000000   
5  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   0.0  0.608313   
6  0.000000  0.000000  0.000000  0.519708  0.000000  0.000000   0.0  0.000000   
7  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   0.5  0.000000   

      issue     learn  ...  password   payment     place  powerful  \
0  0.000000  0.000000  ...  0.000000  0.435556  0.000000  0.000000   
1  0.000000  0.000000  ...  0.000000  0.000000  0.000000  0.000000   
2  0.000000  0.000000  ...  0.000000  

In [14]:
# Cosine Similarity:
from sklearn.metrics.pairwise import cosine_similarity

sent1 = "Machine learning is very interesting"
sent2 = "Deep learning is very powerful"
sent3 = "Payment failed during transaction"

three_sentences = [sent1, sent2, sent3]


processed_sentences = [preprocess_text(sentence) for sentence in three_sentences]

#print("Processed Sentences:")
#print(processed_sentences)

# Converting into TF-IDF vectors
sentence_vectors = vectorizer.transform(processed_sentences)


# Calculating cosine similarity matrix
similarity_matrix = cosine_similarity(sentence_vectors)

print("Cosine Similarity Matrix:")
print(similarity_matrix)

Cosine Similarity Matrix:
[[1.         0.25991021 0.        ]
 [0.25991021 1.         0.        ]
 [0.         0.         1.        ]]


In [15]:
# Displaying similarity matrix in DataFrame form

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=three_sentences,
    columns=three_sentences
)

print("Cosine Similarity Between 3 Sentences:")
print(similarity_df)

Cosine Similarity Between 3 Sentences:
                                      Machine learning is very interesting  \
Machine learning is very interesting                               1.00000   
Deep learning is very powerful                                     0.25991   
Payment failed during transaction                                  0.00000   

                                      Deep learning is very powerful  \
Machine learning is very interesting                         0.25991   
Deep learning is very powerful                               1.00000   
Payment failed during transaction                            0.00000   

                                      Payment failed during transaction  
Machine learning is very interesting                                0.0  
Deep learning is very powerful                                      0.0  
Payment failed during transaction                                   1.0  


In [16]:
# Showing pairwise similarity values clearly

print("Sentence 1 vs Sentence 2 Similarity:", similarity_matrix[0][1])
print("Sentence 1 vs Sentence 3 Similarity:", similarity_matrix[0][2])
print("Sentence 2 vs Sentence 3 Similarity:", similarity_matrix[1][2])

Sentence 1 vs Sentence 2 Similarity: 0.2599102118500098
Sentence 1 vs Sentence 3 Similarity: 0.0
Sentence 2 vs Sentence 3 Similarity: 0.0


In [18]:
# Training data for text classification

train_sentences = [
    "Machine learning is interesting",
    "Deep learning is powerful",
    "NLP helps in text processing",
    "Payment failed in transaction",
    "Unable to login in my account",
    "Account hacked and password changed"
]

train_labels = [1, 1, 1, 0, 0, 0]


# Preprocess training sentences
train_processed = [preprocess_text(sentence) for sentence in train_sentences]

print("Processed Training Data:")
print(train_processed)


# Converting into TF-IDF vectors
X_train = vectorizer.fit_transform(train_processed)
y_train = train_labels

Processed Training Data:
['machine learn interest', 'deep learn powerful', 'nlp help text processing', 'payment fail transaction', 'unable login account', 'account hack password change']


In [19]:
# Training Logistic Regression model
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [20]:
# New test sentence
test_sentence = "Artificial intelligence and machine learning are useful"

test_processed = preprocess_text(test_sentence)

# Converting into TF-IDF vector
X_test = vectorizer.transform([test_processed])

# Predicting class
prediction = log_reg.predict(X_test)

print("Test Sentence:", test_sentence)
print("Predicted Class:", prediction[0])

Test Sentence: Artificial intelligence and machine learning are useful
Predicted Class: 1


In [24]:
# Text data for clustering

cluster_sentences = [
    "Machine learning is interesting",
    "Deep learning is powerful",
    "NLP is useful for text processing",
    "Payment failed during transaction",
    "Login issue after password reset",
    "Account hacked and password changed"
]

cluster_processed = [preprocess_text(sentence) for sentence in cluster_sentences]

print("Processed Clustering Data:")
print(cluster_processed)

# Converting into TF-IDF vectors
X_cluster = vectorizer.fit_transform(cluster_processed)

Processed Clustering Data:
['machine learn interest', 'deep learn powerful', 'nlp useful text processing', 'payment fail transaction', 'login issue password reset', 'account hack password change']


In [26]:
# Applying KMeans clustering
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42)
kmeans.fit(X_cluster)

print("Cluster Labels:")
print(kmeans.labels_)

Cluster Labels:
[0 0 0 0 1 1]


In [29]:
# clustering results in DataFrame

cluster_result = pd.DataFrame({
    "Original_Text": cluster_sentences,
    "Processed_Text": cluster_processed,
    "Cluster": kmeans.labels_
})

print("Text Clustering Result:")
print(cluster_result)

Text Clustering Result:
                         Original_Text                Processed_Text  Cluster
0      Machine learning is interesting        machine learn interest        0
1            Deep learning is powerful           deep learn powerful        0
2    NLP is useful for text processing    nlp useful text processing        0
3    Payment failed during transaction      payment fail transaction        0
4     Login issue after password reset    login issue password reset        1
5  Account hacked and password changed  account hack password change        1
